In [2]:
import pandas as pd
import dask.dataframe as dd

In [3]:
tf = pd.read_csv("Trips_Full Data.csv")

In [4]:
td = pd.read_csv("Trips_by_Distance.csv")

In [5]:
td.columns = (td.columns.str.strip().str.lower().str.replace(" ","_"))
td = td.fillna(td.mean(numeric_only=True)) # fills missing numeric values with its mean 

In [6]:
# handles missing categorical values
td["state_postal_code"] = td["state_postal_code"].fillna("Unknown")
td["county_name"] = td["county_name"].fillna("Unknown")

In [7]:
tf.columns = (tf.columns.str.strip().str.lower().str.replace(" ","_"))  

In [8]:
td.dtypes

level                              object
date                               object
state_fips                        float64
state_postal_code                  object
county_fips                       float64
county_name                        object
population_staying_at_home        float64
population_not_staying_at_home    float64
number_of_trips                   float64
number_of_trips_<1                float64
number_of_trips_1-3               float64
number_of_trips_3-5               float64
number_of_trips_5-10              float64
number_of_trips_10-25             float64
number_of_trips_25-50             float64
number_of_trips_50-100            float64
number_of_trips_100-250           float64
number_of_trips_250-500           float64
number_of_trips_>=500             float64
row_id                             object
week                                int64
month                               int64
dtype: object

In [9]:
td["date"] = pd.to_datetime(td["date"])

In [10]:
tf["date"] = pd.to_datetime(tf["date"])

In [25]:
# axis one name in the dataframe
Distance_by_Miles = [1, 3, 5, 10, 25, 50, 100, 250, 500]

# axis two name in the dataframe
Number_of_Trips = [
    tf["trips_<1_mile"].mean(),
    tf["trips_1-3_miles"].mean(),
    tf["trips_3-5_miles"].mean(),
    tf["trips_5-10_miles"].mean(),
    tf["trips_10-25_miles"].mean(),
    tf["trips_25-50_miles"].mean(),
    tf["trips_50-100_miles"].mean(),
    tf["trips_100-250_miles"].mean(),
    tf["trips_250-500_miles"].mean()
]

# dataframe construction
model_dataframe = pd.DataFrame({
    "Distance": Distance_by_Miles,
    "Trips Number": Number_of_Trips
})

print(model_dataframe)

   Distance  Trips Number
0         1  3.259764e+08
1         3  3.694767e+08
2         5  1.815558e+08
3        10  2.334445e+08
4        25  2.310785e+08
5        50  6.915913e+07
6       100  1.887832e+07
7       250  6.850130e+06
8       500  1.829242e+06


In [26]:
from sklearn.model_selection import train_test_split  # imported library

x = model_dataframe[["Distance"]]            # setting up value
y = model_dataframe["Trips Number"]

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.3, random_state=42
)

print("x_train")
print(x_train)

print("x_test")
print(x_test)

x_train
   Distance
0         1
8       500
2         5
4        25
3        10
6       100
x_test
   Distance
7       250
1         3
5        50


In [27]:
from sklearn.linear_model import LinearRegression

linear_model = LinearRegression()
linear_model.fit(x_train, y_train)

linear_prediction = linear_model.predict(x_test)

print("Intercept =", linear_model.intercept_)
print("Slope =", linear_model.coef_[0])
print("Linear predictions =", linear_prediction)

Intercept = 217642001.73787186
Slope = -488438.80499010877
Linear predictions = [9.55323005e+07 2.16176685e+08 1.93220061e+08]


In [28]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2)

x_train_poly = poly.fit_transform(x_train)
x_test_poly = poly.transform(x_test)

poly_model = LinearRegression()
poly_model.fit(x_train_poly, y_train)

poly_prediction = poly_model.predict(x_test_poly)

print("Polynomial predictions =", poly_prediction)

Polynomial predictions = [-1.67431093e+08  2.62690651e+08  1.35112222e+08]
